# DQN VANILLA

## Prueba de models.py

In [1]:
import sys
import torch

sys.path.append("../src")

from models import DQN

N_ACCIONES = 6
SEMILLA = 42

torch.manual_seed(SEMILLA)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

modelo = DQN(n_acciones=N_ACCIONES).to(device)

#simular un batch de dos observaciones preprocesadas
observaciones_prueba = torch.randint(
    low=0,
    high=256,
    size=(2, 4, 84, 84),
    dtype=torch.uint8,
    device=device,
)

with torch.no_grad():
    valores_q = modelo(observaciones_prueba)

numero_parametros = sum(
    parametro.numel()
    for parametro in modelo.parameters()
)

print(f"Dispositivo: {device}")
print(f"Forma de entrada: {observaciones_prueba.shape}")
print(f"Forma de salida: {valores_q.shape}")
print(f"Número de parámetros: {numero_parametros:,}")
print("\nValores Q de prueba:")
print(valores_q)

assert valores_q.shape == (2, N_ACCIONES)
print("\nLa arquitectura produce las dimensiones correctas.")

Dispositivo: mps
Forma de entrada: torch.Size([2, 4, 84, 84])
Forma de salida: torch.Size([2, 6])
Número de parámetros: 1,687,206

Valores Q de prueba:
tensor([[ 0.0458, -0.0209, -0.0371, -0.0343, -0.0102,  0.0209],
        [ 0.0580, -0.0216, -0.0406, -0.0354, -0.0103,  0.0161]],
       device='mps:0')

La arquitectura produce las dimensiones correctas.


## Prueba de replay_buffer.py

In [2]:
from replay_buffer import ReplayBuffer

buffer_prueba = ReplayBuffer(
    capacidad=10,
    forma_observacion=(4, 84, 84),
    seed=SEMILLA,
)

#agregar más experiencias que su capacidad para comprobar que reemplaza correctamente las experiencias antiguas
for paso in range(12):
    observacion = torch.randint(
        0,
        256,
        size=(4, 84, 84),
        dtype=torch.uint8,
    ).numpy()

    siguiente_observacion = torch.randint(
        0,
        256,
        size=(4, 84, 84),
        dtype=torch.uint8,
    ).numpy()

    buffer_prueba.agregar(
        observacion=observacion,
        accion=paso % N_ACCIONES,
        recompensa=float(paso),
        siguiente_observacion=siguiente_observacion,
        finalizado=(paso % 5 == 0),
    )

batch = buffer_prueba.muestrear(
    batch_size=4,
    device=device,
)

(
    observaciones_batch,
    acciones_batch,
    recompensas_batch,
    siguientes_observaciones_batch,
    finalizados_batch,
) = batch

print(f"Tamaño actual del buffer: {len(buffer_prueba)}")
print(f"Capacidad máxima: {buffer_prueba.capacidad}")
print(
    f"Memoria aproximada: "
    f"{buffer_prueba.memoria_aproximada_mb:.2f} MB"
)

print("\nFORMAS DEL BATCH")
print(f"Observaciones: {observaciones_batch.shape}")
print(f"Acciones: {acciones_batch.shape}")
print(f"Recompensas: {recompensas_batch.shape}")
print(
    "Siguientes observaciones: "
    f"{siguientes_observaciones_batch.shape}"
)
print(f"Finalizados: {finalizados_batch.shape}")

print("\nTIPOS DE DATOS")
print(f"Observaciones: {observaciones_batch.dtype}")
print(f"Acciones: {acciones_batch.dtype}")
print(f"Recompensas: {recompensas_batch.dtype}")
print(f"Finalizados: {finalizados_batch.dtype}")

assert len(buffer_prueba) == 10
assert observaciones_batch.shape == (4, 4, 84, 84)
assert acciones_batch.shape == (4,)
assert recompensas_batch.shape == (4,)
assert finalizados_batch.shape == (4,)

print("\nEl replay buffer funciona correctamente.")

Tamaño actual del buffer: 10
Capacidad máxima: 10
Memoria aproximada: 0.54 MB

FORMAS DEL BATCH
Observaciones: torch.Size([4, 4, 84, 84])
Acciones: torch.Size([4])
Recompensas: torch.Size([4])
Siguientes observaciones: torch.Size([4, 4, 84, 84])
Finalizados: torch.Size([4])

TIPOS DE DATOS
Observaciones: torch.uint8
Acciones: torch.int64
Recompensas: torch.float32
Finalizados: torch.bool

El replay buffer funciona correctamente.


## Prueba epsilon-greedy

In [3]:
import importlib
import train
import numpy as np

importlib.reload(train)

from train import (
    ConfigDQN,
    calcular_epsilon,
    seleccionar_accion,
)

config = ConfigDQN()

# Verificar el decaimiento de epsilon en momentos importantes
pasos_prueba = [
    0,
    50_000,
    100_000,
    150_000,
    200_000,
    250_000,
    500_000,
]

print("DECAIMIENTO DE EPSILON")

for paso in pasos_prueba:
    epsilon = calcular_epsilon(paso, config)
    print(f"Paso {paso:>7,}: epsilon = {epsilon:.3f}")

rng_prueba = np.random.default_rng(SEMILLA)

acciones_aleatorias = [
    seleccionar_accion(
        modelo=modelo,
        observacion=observaciones_prueba[0].cpu().numpy(),
        epsilon=1.0,
        n_acciones=N_ACCIONES,
        device=device,
        rng=rng_prueba,
    )
    for _ in range(20)
]

acciones_greedy = [
    seleccionar_accion(
        modelo=modelo,
        observacion=observaciones_prueba[0].cpu().numpy(),
        epsilon=0.0,
        n_acciones=N_ACCIONES,
        device=device,
        rng=rng_prueba,
    )
    for _ in range(5)
]

print("\nAcciones con epsilon = 1.0:")
print(acciones_aleatorias)

print("\nAcciones con epsilon = 0.0:")
print(acciones_greedy)

assert len(set(acciones_greedy)) == 1
assert all(0 <= accion < N_ACCIONES for accion in acciones_aleatorias)

print("\nLa estrategia epsilon-greedy funciona correctamente.")

DECAIMIENTO DE EPSILON
Paso       0: epsilon = 1.000
Paso  50,000: epsilon = 0.820
Paso 100,000: epsilon = 0.640
Paso 150,000: epsilon = 0.460
Paso 200,000: epsilon = 0.280
Paso 250,000: epsilon = 0.100
Paso 500,000: epsilon = 0.100

Acciones con epsilon = 1.0:
[3, 2, 1, 0, 4, 4, 3, 2, 2, 4, 0, 3, 1, 3, 0, 5, 4, 1, 3, 0]

Acciones con epsilon = 0.0:
[0, 0, 0, 0, 0]

La estrategia epsilon-greedy funciona correctamente.


## Prueba de actualización de DQN

In [4]:
import copy

importlib.reload(train)

from train import actualizar_modelo

#crear modelos nuevos 
modelo_online_prueba = DQN(n_acciones=N_ACCIONES).to(device)
modelo_target_prueba = copy.deepcopy(modelo_online_prueba).to(device)

modelo_target_prueba.eval()

buffer_actualizacion = ReplayBuffer(
    capacidad=64,
    forma_observacion=(4, 84, 84),
    seed=SEMILLA,
)

rng_actualizacion = np.random.default_rng(SEMILLA)

#crear experiencias artificiales para probar la operación matemática
for _ in range(64):
    observacion = rng_actualizacion.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    siguiente_observacion = rng_actualizacion.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    buffer_actualizacion.agregar(
        observacion=observacion,
        accion=int(rng_actualizacion.integers(N_ACCIONES)),
        recompensa=float(rng_actualizacion.choice([-1.0, 0.0, 1.0])),
        siguiente_observacion=siguiente_observacion,
        finalizado=bool(rng_actualizacion.random() < 0.1),
    )

optimizador_prueba = torch.optim.Adam(
    modelo_online_prueba.parameters(),
    lr=config.learning_rate,
)

#guardar una copia de los pesos antes de actualizar
pesos_antes = {
    nombre: parametro.detach().clone()
    for nombre, parametro in modelo_online_prueba.named_parameters()
}

metricas_actualizacion = actualizar_modelo(
    modelo_online=modelo_online_prueba,
    modelo_target=modelo_target_prueba,
    replay_buffer=buffer_actualizacion,
    optimizador=optimizador_prueba,
    config=config,
    device=device,
    usar_double_dqn=False,
)

parametros_modificados = sum(
    not torch.equal(
        pesos_antes[nombre],
        parametro.detach(),
    )
    for nombre, parametro in modelo_online_prueba.named_parameters()
)

print("MÉTRICAS DE LA ACTUALIZACIÓN")
print(f"Loss: {metricas_actualizacion['loss']:.6f}")
print(
    f"Valor Q promedio: "
    f"{metricas_actualizacion['q_promedio']:.6f}"
)
print(
    f"Target promedio: "
    f"{metricas_actualizacion['target_promedio']:.6f}"
)
print(
    f"Tensores de parámetros modificados: "
    f"{parametros_modificados}"
)

assert np.isfinite(metricas_actualizacion["loss"])
assert parametros_modificados > 0

print("\nLa actualización del DQN funciona correctamente.")

MÉTRICAS DE LA ACTUALIZACIÓN
Loss: 0.358199
Valor Q promedio: 0.001445
Target promedio: 0.162015
Tensores de parámetros modificados: 10

La actualización del DQN funciona correctamente.


## Prueba evaluación greedy

In [5]:
import evaluation

importlib.reload(evaluation)

from evaluation import evaluar_modelo

print("Evaluando el modelo sin entrenamiento...\n")

resultados_sin_entrenar, resumen_sin_entrenar = evaluar_modelo(
    modelo=modelo,
    config=config,
    device=device,
    n_episodios=2,
    seed_base=1000,
)

print("RESULTADOS POR EPISODIO")

for resultado in resultados_sin_entrenar:
    print(
        f"Episodio {resultado['episodio']} | "
        f"seed={resultado['seed']} | "
        f"recompensa={resultado['recompensa_total']:.2f} | "
        f"pasos={resultado['pasos']}"
    )

print("\nRESUMEN")
for metrica, valor in resumen_sin_entrenar.items():
    print(f"{metrica}: {valor:.2f}")

Evaluando el modelo sin entrenamiento...



A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


RESULTADOS POR EPISODIO
Episodio 0 | seed=1000 | recompensa=0.00 | pasos=686
Episodio 1 | seed=1001 | recompensa=0.00 | pasos=685

RESUMEN
promedio: 0.00
mediana: 0.00
desviacion: 0.00
minimo: 0.00
maximo: 0.00


## Memoria disponible y costo del buffer

In [6]:
import psutil

memoria_total_gb = psutil.virtual_memory().total / (1024 ** 3)
memoria_disponible_gb = psutil.virtual_memory().available / (1024 ** 3)

def estimar_memoria_buffer(capacidad, forma=(4, 84, 84)):
    bytes_observacion = np.prod(forma) * np.dtype(np.uint8).itemsize

    bytes_por_transicion = (
        2 * bytes_observacion
        + np.dtype(np.int64).itemsize
        + np.dtype(np.float32).itemsize
        + np.dtype(np.bool_).itemsize
    )

    return capacidad * bytes_por_transicion / (1024 ** 3)

print(f"Memoria RAM total: {memoria_total_gb:.2f} GB")
print(f"Memoria RAM disponible ahora: {memoria_disponible_gb:.2f} GB")

print("\nMEMORIA ESTIMADA DEL REPLAY BUFFER")

for capacidad in [10_000, 20_000, 50_000, 100_000]:
    memoria = estimar_memoria_buffer(capacidad)
    print(f"{capacidad:>7,} transiciones: {memoria:.2f} GB")

Memoria RAM total: 16.00 GB
Memoria RAM disponible ahora: 2.84 GB

MEMORIA ESTIMADA DEL REPLAY BUFFER
 10,000 transiciones: 0.53 GB
 20,000 transiciones: 1.05 GB
 50,000 transiciones: 2.63 GB
100,000 transiciones: 5.26 GB


## Smoke test del entrenamiento

In [7]:
importlib.reload(train)

from train import ConfigDQN, entrenar_dqn

config_prueba = ConfigDQN(
    nombre_experimento="smoke_test_dqn",
    total_pasos=1_000,
    capacidad_buffer=1_000,
    inicio_entrenamiento=200,
    batch_size=32,
    frecuencia_entrenamiento=4,
    frecuencia_actualizacion_target=500,
    frecuencia_log=250,
    frecuencia_evaluacion=1_000,
    episodios_evaluacion=1,
    pasos_decay_epsilon=1_000,
)

resultado_prueba = entrenar_dqn(
    config=config_prueba,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print("\nSMOKE TEST FINALIZADO")
print(
    "Mejor promedio de evaluación: "
    f"{resultado_prueba['mejor_promedio_evaluacion']:.2f}"
)
print(
    f"Modelo guardado en: "
    f"{resultado_prueba['carpeta_modelo']}"
)
print(
    f"Logs guardados en: "
    f"{resultado_prueba['carpeta_logs']}"
)

Paso 250/1,000 | episodio=1 | epsilon=0.775 | loss=0.0001 | Q=0.058
Paso 500/1,000 | episodio=2 | epsilon=0.550 | loss=0.0304 | Q=0.060
Paso 750/1,000 | episodio=3 | epsilon=0.325 | loss=0.0148 | Q=0.096
Paso 1,000/1,000 | episodio=5 | epsilon=0.100 | loss=0.0002 | Q=0.086

EVALUACIÓN | paso=1,000 | promedio=90.00 | mediana=90.00 | máximo=90.00


SMOKE TEST FINALIZADO
Mejor promedio de evaluación: 90.00
Modelo guardado en: ../models/smoke_test_dqn
Logs guardados en: ../logs/entrenamientos/smoke_test_dqn


In [11]:
from pathlib import Path
import pandas as pd

ruta_updates_prueba = Path(
    "../logs/entrenamientos/smoke_test_dqn/actualizaciones.csv"
)

df_updates_prueba = pd.read_csv(ruta_updates_prueba)

ultimo_registro = (
    df_updates_prueba
    .sort_values("paso_global")
    .iloc[-1]
)

pasos_realizados = int(ultimo_registro["paso_global"])
segundos_transcurridos = float(
    ultimo_registro["tiempo_segundos"]
)

pasos_por_segundo = (
    pasos_realizados / segundos_transcurridos
)

horas_estimadas_500k = (
    500_000 / pasos_por_segundo / 3600
)

print(f"Pasos realizados: {pasos_realizados:,}")
print(
    f"Tiempo del smoke test: "
    f"{segundos_transcurridos / 60:.2f} minutos"
)
print(
    f"Velocidad aproximada: "
    f"{pasos_por_segundo:.2f} pasos/segundo"
)
print(
    f"Tiempo estimado para 500,000 pasos: "
    f"{horas_estimadas_500k:.2f} horas"
)

print("\nARCHIVOS DEL SMOKE TEST")

for nombre in [
    "config.json",
    "ultimo_checkpoint.pt",
    "mejor_modelo.pt",
    "checkpoint_final.pt",
]:
    ruta = Path("../models/smoke_test_dqn") / nombre
    print(f"{nombre}: {'OK' if ruta.exists() else 'NO ENCONTRADO'}")

Pasos realizados: 1,000
Tiempo del smoke test: 0.06 minutos
Velocidad aproximada: 275.65 pasos/segundo
Tiempo estimado para 500,000 pasos: 0.50 horas

ARCHIVOS DEL SMOKE TEST
config.json: OK
ultimo_checkpoint.pt: OK
mejor_modelo.pt: OK
checkpoint_final.pt: OK
